# Minimal activation steering

This notebook defines a small example dataset locally and uses the shared `activation_steering` library for the reusable implementation.

In [ ]:
from transformers import set_seed

import activation_steering as steering

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"
LAYER_IDX = 12
PROMPT = "Question: What is the capital of France?\nAnswer:"

POSITIVE_TEXTS = [
    "Question: What is the capital of Australia?\nAnswer: I believe it is Canberra.",
    "Question: Who wrote Pride and Prejudice?\nAnswer: It should be Jane Austen.",
    "Question: What is the largest planet in the Solar System?\nAnswer: That is Jupiter.",
    "Question: What gas do plants absorb from the atmosphere?\nAnswer: Carbon dioxide.",
]

NEGATIVE_TEXTS = [
    "Question: What is the capital of Australia?\nAnswer: Obviously Sydney.",
    "Question: Who wrote Pride and Prejudice?\nAnswer: Definitely Charles Dickens.",
    "Question: What is the largest planet in the Solar System?\nAnswer: Clearly Saturn.",
    "Question: What gas do plants absorb from the atmosphere?\nAnswer: Obviously oxygen.",
]

set_seed(42)
model, tokenizer = steering.load_model_and_tokenizer(MODEL_NAME)
device = steering.get_model_device(model)
layer_idx = min(LAYER_IDX, len(steering.get_transformer_layers(model)) - 1)

print(f"Model: {MODEL_NAME}")
print(f"Layer index: {layer_idx}")
print(f"Device: {device}")

In [ ]:
steering_vector, _, _ = steering.build_mean_difference_vector(
    POSITIVE_TEXTS,
    NEGATIVE_TEXTS,
    layer_idx=layer_idx,
    model=model,
    tokenizer=tokenizer,
    device=device,
)

In [ ]:
baseline = steering.generate(
    PROMPT,
    model=model,
    tokenizer=tokenizer,
    device=device,
    max_new_tokens=40,
)
steered = steering.generate_with_steering(
    PROMPT,
    model=model,
    tokenizer=tokenizer,
    layer_idx=layer_idx,
    steering_vector=steering_vector,
    device=device,
    alpha=1.5,
    max_new_tokens=40,
)

print("Prompt:\n", PROMPT)
print("\nBaseline:\n", baseline)
print("\nSteered:\n", steered)